In [ ]:
# Install required packages
! pip install -q transformers accelerate torch pillow qwen-vl-utils python-dotenv

In [ ]:
# Import libraries
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple
from collections import Counter
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image
import time

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Load Model
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import torch

MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"

print(f"Loading {MODEL_NAME}...")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("✓ Model loaded successfully!")

Loading Qwen/Qwen3-VL-8B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

✓ Model loaded successfully!


In [ ]:
# Functions to load data

def load_albums_dataset(json_path: str, base_folder: str) -> List[Dict]:
    """Load albums from JSON with absolute paths for covers."""
    with open(json_path, 'r') as f:
        albums = json.load(f)

    # Update cover paths to absolute
    for album in albums:
        if album.get('cover_path'):
            album['cover_path'] = os.path.join(base_folder, album['cover_path'])

    return albums

def sample_albums_by_genre(albums: List[Dict], n_per_genre: int = 1, target_genres: List[str] = None) -> List[Dict]:
    """
    Sample albums ensuring diversity across genres.

    Args:
        albums: List of album dictionaries
        n_per_genre: Number of albums to sample per genre
        target_genres: List of specific genres to sample (if None, uses top genres)

    Returns:
        List of sampled albums
    """
    from collections import defaultdict
    import random

    # Group albums by primary genre
    genre_albums = defaultdict(list)
    for album in albums:
        if album.get('metadata', {}).get('all_genres'):
            primary_genre = album['metadata']['all_genres'][0]
            genre_albums[primary_genre].append(album)

    # Determine which genres to sample
    if target_genres is None:
        # Use genres with most albums
        target_genres = sorted(genre_albums.keys(), key=lambda g: len(genre_albums[g]), reverse=True)

    sampled = []
    for genre in target_genres:
        if genre in genre_albums and len(genre_albums[genre]) > 0:
            # Sample n_per_genre albums from this genre
            sample_size = min(n_per_genre, len(genre_albums[genre]))
            sampled.extend(random.sample(genre_albums[genre], sample_size))

    return sampled

def format_track_features_for_llm(tracks: List[Dict]) -> str:
    """Format individual track audio features with raw numerical values."""
    if not tracks:
        return "No audio features available."

    descriptions = []

    for track in tracks:
        track_num = track.get('track_number', '?')
        track_name = track.get('track_name', 'Unknown')

        # Extract features (all raw values, no interpretation)
        energy = track.get('energy', 'N/A')
        tempo = track.get('tempo', 'N/A')
        danceability = track.get('danceability', 'N/A')
        acousticness = track.get('acousticness', 'N/A')
        instrumentalness = track.get('instrumentalness', 'N/A')
        happiness = track.get('happiness', 'N/A')
        liveness = track.get('liveness', 'N/A')
        speechiness = track.get('speechiness', 'N/A')

        # Format track description with raw numbers only
        track_desc = (
            f"Track {track_num} - \"{track_name}\": "
            f"energy={energy}, "
            f"danceability={danceability}, "
            f"acousticness={acousticness}, "
            f"instrumentalness={instrumentalness}, "
            f"happiness={happiness}, "
            f"liveness={liveness}, "
            f"speechiness={speechiness}, "
            f"tempo={tempo} BPM"
        )

        descriptions.append(track_desc)

    return "\n".join(descriptions)

In [ ]:
# Functions for agent setup

TIER_DEFINITIONS = """
Tier 1: Underground/Cult (< 100K plays)
  Small dedicated fanbase, very limited reach

Tier 2: Indie Success (100K - 1M plays)
  Moderate indie following, some streaming presence

Tier 3: Moderate Hit (1M - 10M plays)
  Solid commercial success, genre recognition

Tier 4: Major Success (10M - 100M plays)
  Mainstream hit, widespread recognition

Tier 5: Cultural Phenomenon (> 100M plays)
  Legendary status, multi-platinum level
"""

def get_musicologist_prompt(album: Dict, round_num: int, conversation_history: List = None, is_final_round: bool = False) -> str:
    """Musicologist analyzes audio features (text-only)."""

    genres = ', '.join(album['metadata']['all_genres'][:3])
    description = album['metadata'].get('description', '')

    # Format audio features
    audio_text = format_track_features_for_llm(album['audio_features'].get('tracks', []))

    # Add summary stats for context
    summary = album['audio_features'].get('summary_stats', {})
    energy_stats = summary.get('energy', {})
    if energy_stats:
        audio_text += f"\n\nAlbum Energy Range: {energy_stats.get('min')} - {energy_stats.get('max')} (std={energy_stats.get('std')})"

    prompt = f"""You are a professional musicologist evaluating albums for commercial success prediction.

Genres: {genres}
{f'Description: {description}' if description else ''}

Audio Features Scale: energy, danceability, acousticness, instrumentalness, happiness, liveness, speechiness are on a 0-100 scale.

Audio Analysis (First 3 Tracks):
{audio_text}

{TIER_DEFINITIONS}

Round {round_num}/3:"""

    if conversation_history:
        prompt += f"\n\nPrevious Round Discussions:\n{conversation_history}"

    if is_final_round:
        prompt += """

Based on the musical characteristics, rank all 5 tiers from most likely to least likely.
Provide your response in this EXACT format:
RANKING: [Rank all 5 tiers from most likely to least likely, e.g., "3, 4, 2, 5, 1"]
ARGUMENT: [Your 100-word reasoning focusing on musical merit and production quality]"""
    else:
        prompt += """

Based on the musical characteristics, what tier do you predict?
Provide your response in this EXACT format:
TIER: [number 1-5]
ARGUMENT: [Your 100-word reasoning focusing on musical merit and production quality]"""

    return prompt

def get_market_analyst_prompt(album: Dict, round_num: int, conversation_history: List = None, is_final_round: bool = False) -> Tuple[str, str]:
    """Market Analyst analyzes album cover + text (VLM with image)."""

    genres = ', '.join(album['metadata']['all_genres'][:3])
    year = album['metadata'].get('release_year', 'Unknown')
    description = album['metadata'].get('description', '')

    cover_path = album.get('cover_path', '')

    prompt = f"""You are a market research analyst evaluating albums for commercial success prediction.

Genres: {genres}
Release Year: {year}
{f'Description: {description}' if description else ''}

[Album Cover Image Provided]

{TIER_DEFINITIONS}

Round {round_num}/3:"""

    if conversation_history:
        prompt += f"\n\nPrevious Round Discussions:\n{conversation_history}"

    if is_final_round:
        prompt += """

Based on the visual branding, artist positioning, and market appeal, rank all 5 tiers from most likely to least likely.
Provide your response in this EXACT format:
RANKING: [Rank all 5 tiers from most likely to least likely, e.g., "3, 4, 2, 5, 1"]
ARGUMENT: [Your 100-word reasoning focusing on commercial appeal and market trends]"""
    else:
        prompt += """

Based on the visual branding, artist positioning, and market appeal, what tier do you predict?
Provide your response in this EXACT format:
TIER: [number 1-5]
ARGUMENT: [Your 100-word reasoning focusing on commercial appeal and market trends]"""

    return prompt, cover_path

def get_ar_expert_prompt(album: Dict, round_num: int, conversation_history: str, is_final_round: bool = False) -> str:
    """A&R Expert synthesizes all perspectives (text-only)."""

    description = album['metadata'].get('description', '')

    prompt = f"""You are an A&R executive making the final decision on album commercial success prediction.

{f'Description: {description}' if description else ''}

{TIER_DEFINITIONS}

Round {round_num}/3:

Previous Evaluations:
{conversation_history}

Considering both artistic merit (Musicologist) and commercial appeal (Market Analyst), what is your final tier prediction?"""

    if is_final_round:
        prompt += """
Provide your response in this EXACT format:
RANKING: [Rank all 5 tiers from most likely to least likely, e.g., "3, 4, 2, 5, 1"]
ARGUMENT: [Your 100-word reasoning synthesizing both perspectives]"""
    else:
        prompt += """
Provide your response in this EXACT format:
TIER: [number 1-5]
ARGUMENT: [Your 100-word reasoning synthesizing both perspectives]"""

    return prompt

In [ ]:
# Functions for inference and deliberation

def query_model_text_only(prompt: str, max_tokens: int = 150) -> str:
    """Query model with text-only input."""
    messages = [
        {"role": "system", "content": "You are an expert music industry professional."},
        {"role": "user", "content": prompt}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False
        )

    response = processor.batch_decode(outputs, skip_special_tokens=True)[0]

    # Extract response after the prompt
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()

    return response

def query_model_with_image(prompt: str, image_path: str, max_tokens: int = 150) -> str:
    """Query model with image + text input."""
    try:
        # Load image
        image = Image.open(image_path).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=False
            )

        response = processor.batch_decode(outputs, skip_special_tokens=True)[0]

        # Extract response after the prompt
        if "assistant" in response.lower():
            response = response.split("assistant")[-1].strip()

        return response

    except Exception as e:
        print(f"Error with image: {e}")
        return query_model_text_only(prompt, max_tokens)

def parse_agent_response(response: str, expect_ranking: bool = False) -> Dict:
    """Parse agent response to extract tier/ranking and argument."""
    tier = None
    ranking = None
    argument = ""

    lines = response.split('\n')
    for line in lines:
        if line.strip().startswith("TIER:"):
            try:
                tier = int(line.split("TIER:")[1].strip().split()[0])
            except:
                pass
        elif line.strip().startswith("RANKING:"):
            try:
                ranking_str = line.split("RANKING:")[1].strip()
                # Parse comma-separated ranking (e.g., "3, 4, 2, 5, 1")
                ranking = [int(x.strip()) for x in ranking_str.split(',')]
            except:
                pass
        elif line.strip().startswith("ARGUMENT:"):
            argument = line.split("ARGUMENT:")[1].strip()

    # Fallback: search for tier number if not expecting ranking
    if not expect_ranking and tier is None:
        import re
        tier_match = re.search(r'(?:tier|TIER)\s*[:=]?\s*(\d)', response)
        if tier_match:
            tier = int(tier_match.group(1))

    # Fallback argument
    if not argument:
        argument = response[:200]  # Take first 200 chars

    result = {"argument": argument}
    if expect_ranking:
        result["ranking"] = ranking
        result["tier"] = ranking[0] if ranking else None  # Top choice
    else:
        result["tier"] = tier

    return result

def ranked_choice_voting(rankings: List[List[int]]) -> int:
    """
    Perform ranked choice voting (instant runoff) to determine final tier.

    Args:
        rankings: List of rankings from each agent (e.g., [[3,4,2,5,1], [3,2,4,1,5], [4,3,2,5,1]])

    Returns:
        Winning tier (1-5)
    """
    # Remove invalid rankings
    valid_rankings = [r for r in rankings if r and len(r) == 5]

    if not valid_rankings:
        return 3  # Default to middle tier

    # Start with everyone's first choice
    tiers = list(range(1, 6))  # Tiers 1-5

    while len(tiers) > 1:
        # Count first-place votes for remaining tiers
        votes = Counter()
        for ranking in valid_rankings:
            # Find first choice among remaining tiers
            for tier in ranking:
                if tier in tiers:
                    votes[tier] += 1
                    break

        # Check if any tier has majority
        total_votes = len(valid_rankings)
        for tier, count in votes.items():
            if count > total_votes / 2:
                return tier

        # No majority: eliminate tier with fewest votes
        if votes:
            min_votes = min(votes.values())
            # Find all tiers with minimum votes
            to_eliminate = [t for t, v in votes.items() if v == min_votes]
            # Eliminate the one that appears last in most rankings
            for tier in to_eliminate:
                if tier in tiers:
                    tiers.remove(tier)
                    break
        else:
            break

    # Return remaining tier or first choice of first voter
    return tiers[0] if tiers else valid_rankings[0][0]

def run_three_round_deliberation(album: Dict) -> Dict:
    """Run 3 rounds of agent deliberation with ranked choice voting."""

    print(f"\n{'='*70}")
    print(f"Album: {album['metadata']['artist']} - {album['metadata']['title']}")
    print(f"Ground Truth: Tier {album['ground_truth']['tier']}")
    print(f"{'='*70}")

    conversation_history = []

    for round_num in range(1, 4):
        print(f"\n--- Round {round_num} ---")

        round_responses = {}
        is_final = (round_num == 3)

        # Format conversation history for prompts
        history_text = ""
        for past_round in conversation_history:
            history_text += f"\nRound {past_round['round']}:\n"
            history_text += f"  Musicologist: Tier {past_round['musicologist']['tier']} - {past_round['musicologist']['argument']}\n"
            history_text += f"  Market Analyst: Tier {past_round['market_analyst']['tier']} - {past_round['market_analyst']['argument']}\n"
            history_text += f"  A&R Expert: Tier {past_round['ar_expert']['tier']} - {past_round['ar_expert']['argument']}\n"

        # 1. Musicologist (text + audio features)
        print("  Musicologist evaluating...")
        musicologist_prompt = get_musicologist_prompt(album, round_num, history_text, is_final_round=is_final)
        musicologist_response = query_model_text_only(musicologist_prompt)
        round_responses['musicologist'] = parse_agent_response(musicologist_response, expect_ranking=is_final)
        if is_final and round_responses['musicologist'].get('ranking'):
            print(f"    → Ranking: {round_responses['musicologist']['ranking']}")
        else:
            print(f"    → Tier {round_responses['musicologist']['tier']}")

        time.sleep(1)  # Rate limiting

        # 2. Market Analyst (text + image)
        print("  Market Analyst evaluating...")
        analyst_prompt, cover_path = get_market_analyst_prompt(album, round_num, history_text, is_final_round=is_final)
        analyst_response = query_model_with_image(analyst_prompt, cover_path)
        round_responses['market_analyst'] = parse_agent_response(analyst_response, expect_ranking=is_final)
        if is_final and round_responses['market_analyst'].get('ranking'):
            print(f"    → Ranking: {round_responses['market_analyst']['ranking']}")
        else:
            print(f"    → Tier {round_responses['market_analyst']['tier']}")

        time.sleep(1)  # Rate limiting

        # 3. A&R Expert (text + conversation)
        print("  A&R Expert synthesizing...")
        ar_prompt = get_ar_expert_prompt(album, round_num, history_text, is_final_round=is_final)
        ar_response = query_model_text_only(ar_prompt)
        round_responses['ar_expert'] = parse_agent_response(ar_response, expect_ranking=is_final)
        if is_final and round_responses['ar_expert'].get('ranking'):
            print(f"    → Ranking: {round_responses['ar_expert']['ranking']}")
        else:
            print(f"    → Tier {round_responses['ar_expert']['tier']}")

        # Store round
        conversation_history.append({
            'round': round_num,
            **round_responses
        })

        time.sleep(1)

    # After Round 3: Ranked Choice Voting
    print("\n--- Final Tier Determination (Ranked Choice Voting) ---")

    # Extract rankings from Round 3
    final_rankings = []
    round_3 = conversation_history[2]

    for agent in ['musicologist', 'market_analyst', 'ar_expert']:
        ranking = round_3[agent].get('ranking')
        if ranking:
            final_rankings.append(ranking)
            print(f"  {agent.replace('_', ' ').title()}: {ranking}")

    # Perform ranked choice voting
    if final_rankings:
        final_prediction = ranked_choice_voting(final_rankings)
        print(f"\n  Final Prediction (Ranked Choice): Tier {final_prediction}")
    else:
        # Fallback to simple voting if rankings failed
        final_votes = [
            round_3['musicologist'].get('tier'),
            round_3['market_analyst'].get('tier'),
            round_3['ar_expert'].get('tier')
        ]
        final_votes = [v for v in final_votes if v is not None]

        if final_votes:
            vote_counts = Counter(final_votes)
            final_prediction = vote_counts.most_common(1)[0][0]
            print(f"  WARNING: Using fallback voting. Votes: {final_votes}")
        else:
            final_prediction = 3
            print("  WARNING: No valid votes, defaulting to Tier 3")

    result = {
        'album_id': album['album_id'],
        'album_title': f"{album['metadata']['artist']} - {album['metadata']['title']}",
        'ground_truth': album['ground_truth']['tier'],
        'final_prediction': final_prediction,
        'conversation_history': conversation_history,
        'final_rankings': final_rankings
    }

    return result

In [ ]:
# Functions for evaluation

def evaluate_predictions(results: List[Dict]) -> Dict:
    """Calculate evaluation metrics."""

    correct = 0
    off_by_one = 0
    total = len(results)
    errors = []

    for result in results:
        true_tier = result['ground_truth']
        pred_tier = result['final_prediction']

        error = abs(pred_tier - true_tier)
        errors.append(error)

        if error == 0:
            correct += 1
            off_by_one += 1
        elif error == 1:
            off_by_one += 1

    accuracy = correct / total * 100
    off_by_one_acc = off_by_one / total * 100
    mae = sum(errors) / total

    metrics = {
        'accuracy': accuracy,
        'off_by_one_accuracy': off_by_one_acc,
        'mean_absolute_error': mae,
        'total_albums': total
    }

    return metrics

def print_results(results: List[Dict], metrics: Dict):
    """Print formatted results."""

    print("\n" + "="*70)
    print("PILOT RUN RESULTS")
    print("="*70)

    print(f"\nTotal Albums Evaluated: {metrics['total_albums']}")
    print(f"Accuracy (Exact Match): {metrics['accuracy']:.1f}%")
    print(f"Off-by-One Accuracy: {metrics['off_by_one_accuracy']:.1f}%")
    print(f"Mean Absolute Error: {metrics['mean_absolute_error']:.2f} tiers")

    print("\n" + "="*70)
    print("INDIVIDUAL RESULTS")
    print("="*70)

    for result in results:
        print(f"\n{result['album_title']}")
        print(f"  Ground Truth: Tier {result['ground_truth']}")
        print(f"  Prediction: Tier {result['final_prediction']}")
        if result.get('final_rankings'):
            print(f"  Final Rankings:")
            agent_names = ['Musicologist', 'Market Analyst', 'A&R Expert']
            for i, ranking in enumerate(result['final_rankings']):
                print(f"    {agent_names[i]}: {ranking}")

        # Show convergence across rounds
        print(f"  Round Evolution:")
        for round_data in result['conversation_history']:
            print(f"    Round {round_data['round']}: M={round_data['musicologist']['tier']}, "
                  f"MA={round_data['market_analyst']['tier']}, AR={round_data['ar_expert']['tier']}")

In [ ]:
# Execution

# Configuration
from google.colab import drive
drive.mount('/content/drive')

BASE_FOLDER = "/content/drive/MyDrive/AI Agents/sampled_albums_with_audio"
JSON_PATH = f"{BASE_FOLDER}/albums_dataset.json"

print("Loading albums dataset...")
albums = load_albums_dataset(JSON_PATH, BASE_FOLDER)

print("Loading albums dataset...")
albums = load_albums_dataset(JSON_PATH, BASE_FOLDER)

# Sample 8 albums with genre diversity (1 per genre)
# If you have specific genres in mind, specify them here
# Example: target_genres = ['pop', 'rock', 'hip hop', 'electronic', 'r&b', 'country', 'indie', 'jazz']
pilot_albums = sample_albums_by_genre(albums, n_per_genre=1, target_genres=None)[:8]

print(f"\nRunning pilot on {len(pilot_albums)} albums with genre diversity...")
print("Selected albums:")
for i, album in enumerate(pilot_albums):
    genre = album['metadata']['all_genres'][0] if album['metadata'].get('all_genres') else 'Unknown'
    print(f"  {i+1}. {album['metadata']['artist']} - {album['metadata']['title']} ({genre})")

results = []
for i, album in enumerate(pilot_albums):
    print(f"\n\n{'#'*70}")
    print(f"Processing Album {i+1}/{len(pilot_albums)}")
    print(f"{'#'*70}")

    try:
        result = run_three_round_deliberation(album)
        results.append(result)
    except Exception as e:
        print(f"ERROR processing album: {e}")
        continue

    # Save intermediate results
    with open('/content/pilot_results_intermediate.json', 'w') as f:
        json.dump(results, f, indent=2)

# Evaluate
metrics = evaluate_predictions(results)

# Print results
print_results(results, metrics)

# Save final results
output = {
    'results': results,
    'metrics': metrics
}

with open('/content/pilot_results_final.json', 'w') as f:
    json.dump(output, f, indent=2)

print("\n✓ Results saved to /content/pilot_results_final.json")
print("\nPilot run complete!")


Mounted at /content/drive
Loading albums dataset...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading albums dataset...

Running pilot on 8 albums with genre diversity...
Selected albums:
  1. Halsey - hopeless fountain kingdom (Deluxe) (ambient)
  2. Miley Cyrus - Breakout (pop rock)
  3. Little Mix - LM5 (Deluxe) (hip hop)
  4. Tracy Chapman - Crossroads (pop folk)
  5. Japandroids - Post-Nothing (indie rock)
  6. 3OH!3 - Want (Deluxe) (techno)
  7. Joni Mitchell - Taming the Tiger (folk)
  8. Airbourne - Black Dog Barking (Special Edition) (rock)


######################################################################
Processing Album 1/8
######################################################################

Album: Halsey - hopeless fountain kingdom (Deluxe)
Ground Truth: Tier 4

--- Round 1 ---
  Musicologist evaluating...
    → Tier 4
  Market Analyst evaluating...
    → Tier 4
  A&R Expert synthesizing...
    → Tier 4

--- Round 2 ---
  Musicologist evaluating...
    → Tier 4
  Market Analyst evaluating...
    → Tier 4
  A&R Expert synthesizing...
    → Tier 4

--- Round